In [1]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import ast
import numpy as np
import pydicom
from glob import glob
import nibabel as nib
import dicom2nifti
from pathlib import Path
import shutil

In [2]:
# The idea of this thing is to make a script you can run to get preprocessed data

In [3]:
# PATHS

In [4]:
TRAIN_PATH = "train.csv"
LOCALIZERS_PATH = "train_localizers.csv"
REF_NII_PATH = "/Users/nicolas/Desktop/ref_image.nii.gz"
OUT_PATH = "/Users/nicolas/Desktop/mask1.nii.gz"

In [5]:
## Find series with aneurysms

In [6]:
train_labels = pd.read_csv("train.csv")
train_localizers = pd.read_csv("train_localizers.csv")
train_labels = train_labels[train_labels["Modality"].eq("CTA")]
series_with = train_labels[train_labels["Aneurysm Present"].eq(1)]["SeriesInstanceUID"]

series_root = Path("/Volumes/X9 Pro/aneurysm/series")
uid = series_with.iloc[:40]
p = series_root/uid

In [7]:
desktop = Path.home() / "Desktop"      
dst_root = desktop/"output"
for path in p:
    new_dst = dst_root / path.name
    shutil.copytree(path, new_dst, dirs_exist_ok=True)
    print("Copied to: ", new_dst)

Copied to:  /Users/nicolas/Desktop/output/1.2.826.0.1.3680043.8.498.10005158603912009425635473100344077317
Copied to:  /Users/nicolas/Desktop/output/1.2.826.0.1.3680043.8.498.10022796280698534221758473208024838831
Copied to:  /Users/nicolas/Desktop/output/1.2.826.0.1.3680043.8.498.10030095840917973694487307992374923817
Copied to:  /Users/nicolas/Desktop/output/1.2.826.0.1.3680043.8.498.10034081836061566510187499603024895557
Copied to:  /Users/nicolas/Desktop/output/1.2.826.0.1.3680043.8.498.10035643165968342618460849823699311381
Copied to:  /Users/nicolas/Desktop/output/1.2.826.0.1.3680043.8.498.10042474696169267476037627878420766468
Copied to:  /Users/nicolas/Desktop/output/1.2.826.0.1.3680043.8.498.10077108087009955586144859725246456654
Copied to:  /Users/nicolas/Desktop/output/1.2.826.0.1.3680043.8.498.10086325220791440678552106812785190149
Copied to:  /Users/nicolas/Desktop/output/1.2.826.0.1.3680043.8.498.10092666779602341135460882241562348436
Copied to:  /Users/nicolas/Desktop/ou

In [8]:
# Get the data and labels to the correct format

In [9]:
# Paint sphere:

def paint_sphere_mm(mask3d, cz, cy, cx, r_mm, dz, dy, dx, value=1):
    # convert physical radius to voxel extents
    rz = int(np.ceil(r_mm / dz))
    ry = int(np.ceil(r_mm / dy))
    rx = int(np.ceil(r_mm / dx))

    z0 = max(0, cz - rz)
    z1 = min(mask3d.shape[0] - 1, cz + rz)
    y0 = max(0, cy - ry)
    y1 = min(mask3d.shape[1] - 1, cy + ry)
    x0 = max(0, cx - rx)
    x1 = min(mask3d.shape[2] - 1, cx + rx)

    zz, yy, xx = np.ogrid[z0:z1+1, y0:y1+1, x0:x1+1]

    # squared distance in mm
    dist2 = ((zz - cz) * dz)**2 + ((yy - cy) * dy)**2 + ((xx - cx) * dx)**2
    sph = dist2 <= (r_mm * r_mm)

    mask3d[z0:z1+1, y0:y1+1, x0:x1+1][sph] = value 

In [10]:
str(p.iloc[0])

'/Volumes/X9 Pro/aneurysm/series/1.2.826.0.1.3680043.8.498.10005158603912009425635473100344077317'

In [11]:
for path in p:
# 2) load dicoms for one series and sort
    dicom_files = glob(f"{str(path)}/*.dcm")
    ds_list = [pydicom.dcmread(f, stop_before_pixels=True) for f in dicom_files] # Reads the metadata of all dicom files
    
    # Robust sort that prefers ImagePositionPatient and fallbacks to InstanceNumber
    def sort_key(ds):
        if hasattr(ds, "ImagePositionPatient"):
            return float(ds.ImagePositionPatient[2]) # Z-coordinate of the silice's physical poisition in patient coordinates
        return int(ds.InstanceNumber) # Index identifying the position of the an image within a series 
    
    ds_list.sort(key=sort_key)
    
    # Match each frame to a z coordinate
    sop_to_z = {ds.SOPInstanceUID: i for i, ds in enumerate(ds_list)}
    
    # Get the In-plane spacing (row, col) => dy, dx
    dy, dx = map(float, ds_list[0].PixelSpacing) # Get the physical space between pixel centers in mm/pixel
    
    # Slice spacing: Prefer slice deltas if available (ImagePositionPatient)
    if hasattr(ds_list[0], "SpacingBetweenSlices"):
        dz = float(ds_list[0].SpacingBetweenSlices) # Actual distance of the center (along slice axis) of adjacent slices
    else:
        dz = float(getattr(ds_list[0], "SliceThickness", 1.0))
    
    # This asumes that the volume is a series of dicom files (no multiframe dicom for now)
    
    Z = len(ds_list) # Number of dicom files (number of slices)
    Y = int(ds_list[0].Rows) # Image height in pixels
    X = int(ds_list[0].Columns) # Image width in pixels 
    mask = np.zeros((Z, Y, X), dtype=np.uint8) # Makes a mask of the same size of the image
    
    R_mm = 1.5  # radius in millimeters
    for _, row in train_localizers.iterrows():
        sop = row["SOPInstanceUID"]
        if sop not in sop_to_z:
            continue
        xy = ast.literal_eval(row["coordinates"])
        x = int(round(xy["x"]))
        y = int(round(xy["y"]))
        z = sop_to_z[sop]
        print(x, y, z)
        if 0 <= x < X and 0 <= y < Y and 0 <= z < Z:
            paint_sphere_mm(mask, z, y, x, R_mm, dz, dy, dx, value=1)
            
    # Convert the DICOM series to a reference NIfTI (to get affine/header)
    
    dicom_dir = path
    ref_nii_path = Path.home() / "Desktop" / "refs" / dicom_dir.name / f"{dicom_dir.name}_ref.nii.gz"
    ref_nii_path.parent.mkdir(parents=True, exist_ok=True)  # create subfolder if needed
    
    dicom2nifti.dicom_series_to_nifti(dicom_dir, ref_nii_path, reorient_nifti=True)
    
    ref_img = nib.load(ref_nii_path)
    ref_affine = ref_img.affine
    ref_header = ref_img.header.copy()
    
    mask_xyz = np.transpose(mask, (2, 1, 0))  # Assumes that reorient_nifti=True gives (Z,Y,X) and needs to be transformed to (X,Y,Z)
    
    # Make sure shapes match, otherwise fix ordering/reorient choice
    print("ref shape:", ref_img.shape, "mask shape:", mask_xyz.shape)
    
    # Save mask as NIfTI (uint8 labels)
    mask_img = nib.Nifti1Image(mask_xyz.astype(np.uint8), ref_affine, header=ref_header)
    mask_img.set_data_dtype(np.uint8)
    
    out_path = OUT_PATH
    nib.save(mask_img, out_path)
    print("Saved:", out_path)

258 261 162
ref shape: (512, 512, 276) mask shape: (512, 512, 276)
Saved: /Users/nicolas/Desktop/mask1.nii.gz
195 178 453
ref shape: (512, 512, 671) mask shape: (512, 512, 671)
Saved: /Users/nicolas/Desktop/mask1.nii.gz
208 230 273
ref shape: (512, 512, 451) mask shape: (512, 512, 451)
Saved: /Users/nicolas/Desktop/mask1.nii.gz
250 221 46
ref shape: (512, 512, 83) mask shape: (512, 512, 83)
Saved: /Users/nicolas/Desktop/mask1.nii.gz
224 225 74
289 212 77
233 226 64
ref shape: (512, 512, 228) mask shape: (512, 512, 228)
Saved: /Users/nicolas/Desktop/mask1.nii.gz
254 222 181
ref shape: (512, 512, 587) mask shape: (512, 512, 587)
Saved: /Users/nicolas/Desktop/mask1.nii.gz
228 273 5
248 273 26
ref shape: (30, 584, 584) mask shape: (512, 512, 31)
Saved: /Users/nicolas/Desktop/mask1.nii.gz
256 211 44
ref shape: (512, 512, 82) mask shape: (512, 512, 82)
Saved: /Users/nicolas/Desktop/mask1.nii.gz
272 240 96
ref shape: (512, 512, 281) mask shape: (512, 512, 281)
Saved: /Users/nicolas/Desktop/ma